In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

df = pd.read_csv("/content/train_and_test2.csv")

df.rename(columns={
    "Passengerid": "PassengerId",
    "sibsp": "SibSp",
    "2urvived": "Survived"
}, inplace=True)

df.drop(
    columns=[col for col in df.columns if col.startswith("zero")],
    inplace=True
)

print(df.shape)
print(df.columns)
display(df.head())

lifecycle = {
    "Child": (0, 12),
    "Teenager": (13, 19),
    "Young Adult": (20, 35),
    "Adult": (36, 59),
    "Senior": (60, 120)
}

def get_stage(age):
    if pd.isna(age):
        return "Unknown"

    for stage, ages in lifecycle.items():
        if ages[0] <= age <= ages[1]:
            return stage

    return "Unknown"

df["Lifecycle"] = df["Age"].apply(get_stage)

with PdfPages("titanic_eda_report.pdf") as pdf:

    plt.figure(figsize=(11, 8.5))

    sns.heatmap(
        df.isnull(),
        cbar=False,
        yticklabels=False
    )

    plt.title("Missing Value Heatmap")
    plt.xlabel("Columns")
    plt.ylabel("Passengers")
    plt.tight_layout()

    pdf.savefig()
    plt.close()

    fig, ax = plt.subplots(2, 2, figsize=(11, 8.5))

    sex_survival = df.groupby("Sex")["Survived"].mean() * 100

    sex_survival.plot(
        kind="bar",
        ax=ax[0, 0]
    )

    ax[0, 0].set_title("Survival Rate by Sex")
    ax[0, 0].set_ylabel("Survival Rate (%)")
    ax[0, 0].set_ylim(0, 100)

    class_survival = df.groupby("Pclass")["Survived"].mean() * 100

    class_survival.plot(
        kind="bar",
        ax=ax[0, 1]
    )

    ax[0, 1].set_title("Survival Rate by Class")
    ax[0, 1].set_ylabel("Survival Rate (%)")
    ax[0, 1].set_ylim(0, 100)

    sns.histplot(
        data=df,
        x="Age",
        hue="Survived",
        bins=20,
        kde=True,
        ax=ax[1, 0]
    )

    ax[1, 0].set_title("Age Distribution by Survival")
    ax[1, 0].set_xlabel("Age")
    ax[1, 0].set_ylabel("Count")

    order = [
        "Child",
        "Teenager",
        "Young Adult",
        "Adult",
        "Senior",
        "Unknown"
    ]

    lifecycle_survival = (
        df.groupby("Lifecycle")["Survived"].mean() * 100
    )

    lifecycle_survival = lifecycle_survival.reindex(
        [x for x in order if x in lifecycle_survival.index]
    )

    lifecycle_survival.plot(
        kind="bar",
        ax=ax[1, 1]
    )

    ax[1, 1].set_title("Survival Rate by Lifecycle")
    ax[1, 1].set_ylabel("Survival Rate (%)")
    ax[1, 1].set_ylim(0, 100)

    plt.tight_layout()

    pdf.savefig()
    plt.close()

    plt.figure(figsize=(11, 8.5))
    plt.axis("off")

    text = "Lifecycle Mapping Dictionary\n\n"

    for stage, ages in lifecycle.items():
        text += f"{stage}: {ages[0]} - {ages[1]}\n"

    text += "\nLifecycle Distribution\n\n"

    lifecycle_counts = df["Lifecycle"].value_counts()

    for stage in order:
        if stage in lifecycle_counts.index:
            text += f"{stage}: {lifecycle_counts[stage]}\n"

    text += "\nSurvival by Lifecycle\n\n"

    for stage in lifecycle_survival.index:
        text += f"{stage}: {lifecycle_survival[stage]:.2f}%\n"

    plt.text(
        0.1,
        0.9,
        text,
        fontsize=16,
        verticalalignment="top"
    )

    plt.title("Lifecycle Mapping")
    plt.tight_layout()

    pdf.savefig()
    plt.close()

    correlation = df.select_dtypes(
        include=np.number
    ).corr()

    plt.figure(figsize=(11, 8.5))

    sns.heatmap(
        correlation,
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )

    plt.title("Correlation Heatmap")
    plt.tight_layout()

    pdf.savefig()
    plt.close()

print("titanic_eda_report.pdf created")

(1309, 9)
Index(['PassengerId', 'Age', 'Fare', 'Sex', 'SibSp', 'Parch', 'Pclass',
       'Embarked', 'Survived'],
      dtype='object')


,PassengerId,Age,Fare,Sex,SibSp,Parch,Pclass,Embarked,Survived
0,1,22.0,7.2500,0,1,0,3,2.0,0
1,2,38.0,71.2833,1,1,0,1,0.0,1
2,3,26.0,7.9250,1,0,0,3,2.0,1
3,4,35.0,53.1000,1,1,0,1,2.0,1
4,5,35.0,8.0500,0,0,0,3,2.0,0


titanic_eda_report.pdf created
